# 🎤 Speech Enhancement — Step 3: Denoise a Voice Recording

This notebook:
1. Mounts Google Drive
2. Loads a trained U-Net model from Drive
3. Denoises one or several noisy WAV files
4. Displays before/after spectrograms and lets you play back the audio

### Prerequisites
- Run notebooks 01 and 02 first.
- Your Drive should have `weights/model_best.h5`
- Place your noisy test WAVs under `data/test/` in Drive.

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
%pip install -q librosa soundfile

In [ ]:
# ── 3. Clone repo & add src/ to path ──────────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/YOUR_USERNAME/speech_enhancement_alt.git'  # <── update
REPO_DIR = '/content/speech_enhancement_alt'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready ✓')

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────
import config as C

# Path to the weights to use for denoising
WEIGHTS = os.path.join(C.WEIGHTS_DIR, 'model_best.h5')
print(f'Weights path: {WEIGHTS}')
print(f'Test input dir: {C.AUDIO_INPUT_DIR}')
print(f'Predictions dir: {C.PRED_DIR}')

os.makedirs(C.PRED_DIR, exist_ok=True)

In [ ]:
# ── 5. Upload a noisy WAV directly (optional alternative to Drive) ─────────
# from google.colab import files
# uploaded = files.upload()
# input_file = list(uploaded.keys())[0]   # path of uploaded file
# print('Uploaded:', input_file)

In [ ]:
# ── 6. Denoise a single file ───────────────────────────────────────────────
from predict import denoise_audio

# Change filename to your test file:
input_wav  = os.path.join(C.AUDIO_INPUT_DIR, 'my_noisy_audio.wav')   # ← update
output_wav = os.path.join(C.PRED_DIR, 'denoised_output.wav')

audio_out = denoise_audio(
    input_path   = input_wav,
    output_path  = output_wav,
    weights_path = WEIGHTS,
)
print('Denoising complete!')

In [ ]:
# ── 7. Play back & compare ────────────────────────────────────────────────
import IPython.display as ipd
import soundfile as sf

def play(path, label):
    data, sr = sf.read(path)
    print(f'\n🔊  {label}')
    display(ipd.Audio(data, rate=sr))

play(input_wav,  'Input  (noisy)')
play(output_wav, 'Output (denoised)')

In [ ]:
# ── 8. Visualise spectrogram before vs after ───────────────────────────────
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

def load_spec(wav_path, sr=C.SAMPLE_RATE):
    y, _ = librosa.load(wav_path, sr=sr)
    S    = librosa.stft(y, n_fft=C.N_FFT, hop_length=C.HOP_LENGTH_FFT)
    return librosa.amplitude_to_db(np.abs(S), ref=np.max)

S_noisy   = load_spec(input_wav)
S_denoised = load_spec(output_wav)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
librosa.display.specshow(S_noisy,   sr=C.SAMPLE_RATE,
                         hop_length=C.HOP_LENGTH_FFT,
                         x_axis='time', y_axis='hz', ax=axes[0], cmap='magma')
axes[0].set_title('Noisy Voice Spectrogram')

librosa.display.specshow(S_denoised, sr=C.SAMPLE_RATE,
                         hop_length=C.HOP_LENGTH_FFT,
                         x_axis='time', y_axis='hz', ax=axes[1], cmap='magma')
axes[1].set_title('Denoised Voice Spectrogram')

plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Batch denoise all WAVs in test folder ───────────────────────────────
test_files = [f for f in os.listdir(C.AUDIO_INPUT_DIR) if f.endswith('.wav')]
print(f'Found {len(test_files)} test files')

for fname in test_files:
    in_path  = os.path.join(C.AUDIO_INPUT_DIR, fname)
    out_name = 'denoised_' + fname
    out_path = os.path.join(C.PRED_DIR, out_name)
    print(f'Processing {fname} …')
    denoise_audio(in_path, out_path, weights_path=WEIGHTS)
    print(f'  → {out_path}')

print('\nAll files processed!')